# Anonymize → RAG Pipeline Demo

End-to-end walkthrough: raw text files → PII anonymization → vector-store ingestion → RAG query.

**What this notebook covers**

1. **Pipeline config** — reuse `AnonymizationConfig` + `PresidioDetectorConfig` from the toolkit
2. **Sample data** — write a few text files with synthetic PII to a temp directory
3. **Anonymize** — run `anonymize_files_flow` to scrub PII, inspect the `AnonymizeManifest`
4. **Ingest** — run `rag_file_ingestion_flow` on the anonymized output
5. **Query** — search the vector store to confirm anonymized chunks are retrieved
6. **YAML equivalent** — see how the same pipeline maps to `workflows.yaml` profiles

In [9]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)
sys.path.append(".")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Define the pipeline config

We compose two existing config objects from `genai_tk.workflow.anonymization`:

- `PresidioDetectorConfig` — which PII entities to detect
- `AnonymizationConfig` — wraps the detector config + Faker settings

RAG ingestion parameters are passed directly to `rag_file_ingestion_flow` (flat keyword args, no config class needed).

In [10]:
import tempfile
from pathlib import Path

from genai_tk.workflow.anonymization.core import AnonymizationConfig
from genai_tk.workflow.anonymization.presidio_detector import PresidioDetectorConfig

# ---- Directories (self-contained temp dirs for this demo) ----
_tmpdir = tempfile.mkdtemp(prefix="anon_rag_demo_")
BASE_DIR = f"{_tmpdir}/source"
ANON_DIR = f"{_tmpdir}/anonymized"

# ---- Anonymization config (reuses the existing shared class) ----
anon_config = AnonymizationConfig(
    detector=PresidioDetectorConfig(
        analyzed_fields=["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "LOCATION", "CREDIT_CARD"],
    ),
    faker_seed=42,
)

# ---- RAG ingestion params ----
RETRIEVER_NAME = "default"
PATHSPECS = ["**/*.txt", "**/*.md"]
CHUNK_SIZE = 512
BATCH_SIZE = 10

print("Anonymization config:", anon_config)
print(f"Source dir : {BASE_DIR}")
print(f"Anonymized : {ANON_DIR}")

Anonymization config:
AnonymizationConfig(
    detector=PresidioDetectorConfig(
        analyzed_fields=['PERSON', 'EMAIL_ADDRESS', 'PHONE_NUMBER', 'LOCATION', 'CREDIT_CARD'],
        custom_recognizers=[],
        spacy_model='en_core_web_sm',
        language='en',
        enable_spacy=True,
        score_threshold=0.4
    ),
    faker_seed=42,
    faker_locales=['en_US'],
    fuzzy_deanonymize=True,
    fuzzy_threshold=85
)

Source dir : /tmp/anon_rag_demo_yxc17fqa/source

Anonymized : /tmp/anon_rag_demo_yxc17fqa/anonymized

## 2. Create sample files with synthetic PII

We write a few text files that contain names, emails, and phone numbers so we can verify the anonymization step.

In [11]:
source_dir = Path(BASE_DIR)
source_dir.mkdir(parents=True, exist_ok=True)

SAMPLE_FILES = {
    "contract_001.txt": """\
Service Agreement

This agreement is between Alice Johnson (alice.johnson@acme.com, +1-555-123-4567)
and Bob Smith (bob.smith@globex.io, +1-555-987-6543).

Alice Johnson is based in New York. Bob Smith operates from London.
Payment details: credit card 4111 1111 1111 1111, billing address: 123 Main St.
""",
    "support_ticket_42.txt": """\
Support Ticket #42

Customer: Carol White
Email: carol.white@example.org
Phone: (800) 555-0100
Location: San Francisco, CA

Issue: Unable to log into account. Please contact carol.white@example.org for follow-up.
""",
    "meeting_notes.md": """\
# Q2 Planning Meeting

Attendees: David Lee (david@startup.io), Emma Brown (emma.brown@corp.net)

David Lee proposed the new roadmap. Emma Brown will follow up by email at emma.brown@corp.net.
Next meeting in Chicago on 2026-06-15.
""",
}

for filename, content in SAMPLE_FILES.items():
    (source_dir / filename).write_text(content, encoding="utf-8")
    print(f"  Wrote {filename} ({len(content)} chars)")

print(f"\nSource directory: {source_dir}")

Wrote contract_001.txt (304 chars)

Wrote support_ticket_42.txt (213 chars)

Wrote meeting_notes.md (232 chars)

Source directory: /tmp/anon_rag_demo_yxc17fqa/source

## 3. Anonymize files

We call `anonymize_files_flow` directly using `run_flow_ephemeral` — the same helper used by the
workflow CLI.  The flow:

- Detects PII via **Presidio** (backed by spaCy NER)
- Replaces each entity with a **Faker**-generated value
- Writes anonymized copies to `ANON_DIR`
- Saves an `anonymize_manifest.json` for incremental re-runs

The anonymization logic is **shared** with `AnonymizationMiddleware`:  
the same `anonymize_text()` function runs here at ETL time, and inside the agent at inference time.

In [12]:
from genai_tk.workflow.prefect.flows.anonymize_flow import anonymize_files_flow
from genai_tk.workflow.prefect.run import run_flow_ephemeral

anon_manifest = run_flow_ephemeral(
    anonymize_files_flow,
    base_dir=BASE_DIR,
    output_dir=ANON_DIR,
    pathspecs=PATHSPECS,
    batch_size=BATCH_SIZE,
    analyzed_fields=anon_config.detector.analyzed_fields,
    faker_seed=anon_config.faker_seed,
)

print(f"\nProcessed {len(anon_manifest.entries)} files")
for path, entry in anon_manifest.entries.items():
    print(f"  {entry.output_path}  (hash={entry.source_hash[:8]}...)")

2026-05-12 18:42:03.438 | INFO     | genai_tk.workflow.prefect.flows.anonymize_flow:anonymize_files_flow:205 - Found 3 files matching patterns in '/tmp/anon_rag_demo_yxc17fqa/source'
2026-05-12 18:42:03.439 | INFO     | genai_tk.workflow.prefect.flows.anonymize_flow:anonymize_files_flow:226 - Processing 3 files, skipping 0 unchanged
2026-05-12 18:42:03.440 | INFO     | genai_tk.workflow.prefect.flows.anonymize_flow:anonymize_files_flow:234 - Batch 1/1 (3 files)


18:42:03.471 | WARNING | presidio-analyzer - Entity CARDINAL is not mapped to a Presidio entity, but keeping anyway. Add to `NerModelConfiguration.labels_to_ignore` to remove.

18:42:03.475 | WARNING | presidio-analyzer - Entity FAC is not mapped to a Presidio entity, but keeping anyway. Add to `NerModelConfiguration.labels_to_ignore` to remove.

2026-05-12 18:42:03.484 | DEBUG    | genai_tk.workflow.anonymization.core:anonymize_text:144 - Anonymized 9 entities
2026-05-12 18:42:03.487 | SUCCESS  | genai_tk.workflow.prefect.flows.anonymize_flow:anonymize_file_task:129 - Anonymized 7 entities in contract_001.txt
2026-05-12 18:42:03.521 | DEBUG    | genai_tk.workflow.anonymization.core:anonymize_text:144 - Anonymized 8 entities
2026-05-12 18:42:03.524 | SUCCESS  | genai_tk.workflow.prefect.flows.anonymize_flow:anonymize_file_task:129 - Anonymized 5 entities in meeting_notes.md


18:42:03.555 | WARNING | presidio-analyzer - Entity CARDINAL is not mapped to a Presidio entity, but keeping anyway. Add to `NerModelConfiguration.labels_to_ignore` to remove.

...

2026-05-12 18:42:03.568 | DEBUG    | genai_tk.workflow.anonymization.core:anonymize_text:144 - Anonymized 5 entities
2026-05-12 18:42:03.571 | SUCCESS  | genai_tk.workflow.prefect.flows.anonymize_flow:anonymize_file_task:129 - Anonymized 4 entities in support_ticket_42.txt
2026-05-12 18:42:03.579 | SUCCESS  | genai_tk.workflow.prefect.flows.anonymize_flow:anonymize_files_flow:258 - Anonymization complete: 3 processed, 0 skipped, manifest at /tmp/anon_rag_demo_yxc17fqa/anonymized/anonymize_manifest.json


Processed 3 files

contract_001.txt  (hash=d29f48af...)

meeting_notes.md  (hash=f7ca7b19...)

support_ticket_42.txt  (hash=13afc488...)

### 3a. Inspect the anonymized output

Let's verify that PII has been replaced in the output files.

In [13]:
from rich import print
from rich.panel import Panel

anon_dir = Path(ANON_DIR)

for anon_file in sorted(anon_dir.glob("*.txt")) + sorted(anon_dir.glob("*.md")):
    original = (Path(BASE_DIR) / anon_file.name).read_text()
    anonymized = anon_file.read_text()
    print(
        Panel(
            f"[bold]ORIGINAL:[/bold]\n{original}\n\n[bold]ANONYMIZED:[/bold]\n{anonymized}",
            title=anon_file.name,
            expand=False,
        )
    )

╭────────────────────────────────── contract_001.txt ──────────────────────────────────╮
│ ORIGINAL:                                                                            │
│ Service Agreement                                                                    │
│                                                                                      │
│ This agreement is between Alice Johnson (alice.johnson@acme.com, +1-555-123-4567)    │
│ and Bob Smith (bob.smith@globex.io, +1-555-987-6543).                                │
│                                                                                      │
│ Alice Johnson is based in New York. Bob Smith operates from London.                  │
│ Payment details: credit card 4111 1111 1111 1111, billing address: 123 Main St.      │
│                                                                                      │
│                                                                                      │
│ ANONYMIZED:                                                                          │
│ Service Agreement                                                                    │
│                                                                                      │
│ This agreement is between Allison Hill (donaldgarcia@example.net, +1-555-123-4567)   │
│ and Angie Henderson (davisjesse@example.net, +1-555-987-6543).                       │
│                                                                                      │
│ Allison Hill is based in New Jamesside. Angie Henderson operates from Robinsonshire. │
│ Payment details: credit card 4265423511615594074, billing address: 123 Main St.      │
│                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────── support_ticket_42.txt ──────────────────────────────────╮
│ ORIGINAL:                                                                                 │
│ Support Ticket #42                                                                        │
│                                                                                           │
│ Customer: Carol White                                                                     │
│ Email: carol.white@example.org                                                            │
│ Phone: (800) 555-0100                                                                     │
│ Location: San Francisco, CA                                                               │
│                                                                                           │
│ Issue: Unable to log into account. Please contact carol.white@example.org for follow-up.  │
│                                                                                           │
│                                                                                           │
│ ANONYMIZED:                                                                               │
│ Support Ticket #42                                                                        │
│                                                                                           │
│ Customer: Allison Hill                                                                    │
│ Email: donaldgarcia@example.net                                                           │
│ Phone: +1-219-560-0133                                                                    │
│ Location: Robinsonshire, CA                                                               │
│                                                                                           │
│ Issue: Unable to log into account. Please contact donaldgarcia@example.net for follow-up. │
│                                                                                           │
╰───────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── meeting_notes.md ─────────────────────────────────────────────╮
│ ORIGINAL:                                                                                                 │
│ # Q2 Planning Meeting                                                                                     │
│                                                                                                           │
│ Attendees: David Lee (david@startup.io), Emma Brown (emma.brown@corp.net)                                 │
│                                                                                                           │
│ David Lee proposed the new roadmap. Emma Brown will follow up by email at emma.brown@corp.net.            │
│ Next meeting in Chicago on 2026-06-15.                                                                    │
│                                                                                                           │
│                                                                                                           │
│ ANONYMIZED:                                                                                               │
│ # Q2 Planning Meeting                                                                                     │
│                                                                                                           │
│ Attendees: Allison Hill (donaldgarcia@example.net), Angie Henderson (davisjesse@example.net)              │
│                                                                                                           │
│ Allison Hill proposed the new roadmap. Angie Henderson will follow up by email at davisjesse@example.net. │
│ Next meeting in New Jamesside on 2026-06-15.                                                              │
│                                                                                                           │
╰───────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 3b. Direct use of `anonymize_text()` (lower-level)

The standalone helper can be called directly for single strings — useful for testing or
transforming data outside of a full Prefect flow.

In [14]:
from faker import Faker

from genai_tk.workflow.anonymization.core import anonymize_text
from genai_tk.workflow.anonymization.presidio_detector import PresidioDetector

detector = PresidioDetector(config=anon_config.detector)
Faker.seed(anon_config.faker_seed or 0)
faker = Faker(anon_config.faker_locales)

raw = "Please contact Jane Doe at jane.doe@secret.com or call +1-555-000-1234."
result, mapping = anonymize_text(raw, detector=detector, faker=faker)

print(f"Original : {raw}")
print(f"Anonymized: {result}")
print(f"Mapping   : {mapping}")

2026-05-12 18:42:32.046 | DEBUG    | genai_tk.workflow.anonymization.core:anonymize_text:144 - Anonymized 2 entities


Original : Please contact Jane Doe at jane.doe@secret.com or call +1-555-000-1234.

Anonymized: Please contact Allison Hill at donaldgarcia@example.net or call +1-555-000-1234.

Mapping   : {'Jane Doe': 'Allison Hill', 'jane.doe@secret.com': 'donaldgarcia@example.net'}

## 4. Ingest anonymized files into the vector store

We now run `rag_file_ingestion_flow` on the anonymized output directory.  
The vector store will only ever see the fake names/emails — the originals never leave the ETL step.

In [15]:
from genai_tk.workflow.prefect.flows.rag_flow import rag_file_ingestion_flow

rag_stats = run_flow_ephemeral(
    rag_file_ingestion_flow,
    base_dir=ANON_DIR,
    retriever_name=RETRIEVER_NAME,
    max_chunk_tokens=CHUNK_SIZE,
    chunker_name="auto",
    pathspecs=PATHSPECS,
    batch_size=BATCH_SIZE,
)

print("RAG ingestion stats:", rag_stats)

2026-05-12 18:42:37.610 | INFO     | genai_tk.workflow.prefect.flows.rag_flow:rag_file_ingestion_flow:193 - Starting RAG file ingestion from '/tmp/anon_rag_demo_yxc17fqa/anonymized' to retriever 'default'
2026-05-12 18:42:37.613 | INFO     | genai_tk.workflow.prefect.flows.rag_flow:rag_file_ingestion_flow:197 - Found 3 files matching patterns
2026-05-12 18:42:37.656 | DEBUG    | genai_tk.core.embeddings_store:get_vector_store:237 - Created vector store: genai_tk.core.vector_backends.ChromaBackend/embeddings_qwen3_06b => 'in-memory'
2026-05-12 18:42:37.661 | DEBUG    | genai_tk.workflow.prefect.flows.rag_flow:_prepare_files:70 - Found 3 existing file hashes in vector store
2026-05-12 18:42:37.662 | INFO     | genai_tk.workflow.prefect.flows.rag_flow:_prepare_files:81 - Skipping already processed file: /tmp/anon_rag_demo_yxc17fqa/anonymized/contract_001.txt
2026-05-12 18:42:37.664 | INFO     | genai_tk.workflow.prefect.flows.rag_flow:_prepare_files:81 - Skipping already processed file: /

RAG ingestion stats:
{'total_files': 3, 'processed_files': 0, 'skipped_files': 3, 'total_chunks': 0}

## 5. Query the vector store

Retrieve chunks and confirm the anonymized text is what's stored (not the original PII).

In [ ]:
from genai_tk.core.factories.retriever_factory import RetrieverFactory

managed = RetrieverFactory.create(RETRIEVER_NAME)

results = managed.query("service agreement contact email", k=3)

print(f"Top {len(results)} results:\n")
for i, doc in enumerate(results, 1):
    print(f"[{i}] source={doc.metadata.get('source')}")
    print(f"    {doc.page_content[:200]}")
    print()

## 6. YAML equivalent

The `AnonymizationConfig` above maps directly to `workflows.yaml` profiles.  
The two-step `anonymize_and_ingest` workflow is already registered in `config/workflows.yaml`.

### Auto-generate the profile values from the config

In [16]:
import yaml

yaml_profile = {
    "workflow_profiles": {
        "anonymize_and_ingest_docs": {
            "workflow": "anonymize_and_ingest",
            "values": {
                "base_dir": BASE_DIR,
                "anon_dir": ANON_DIR,
                "pathspecs": PATHSPECS,
                **anon_config.model_dump(),
                "retriever_name": RETRIEVER_NAME,
                "chunk_size": CHUNK_SIZE,
                "batch_size": BATCH_SIZE,
            },
        }
    }
}

print(yaml.dump(yaml_profile, default_flow_style=False, sort_keys=False))

workflow_profiles:
  anonymize_and_ingest_docs:
    workflow: anonymize_and_ingest
    values:
      base_dir: /tmp/anon_rag_demo_yxc17fqa/source
      anon_dir: /tmp/anon_rag_demo_yxc17fqa/anonymized
      pathspecs:
      - '**/*.txt'
      - '**/*.md'
      detector:
        analyzed_fields:
        - PERSON
        - EMAIL_ADDRESS
        - PHONE_NUMBER
        - LOCATION
        - CREDIT_CARD
        custom_recognizers: []
        spacy_model: en_core_web_sm
        language: en
        enable_spacy: true
        score_threshold: 0.4
      faker_seed: 42
      faker_locales:
      - en_US
      fuzzy_deanonymize: true
      fuzzy_threshold: 85
      retriever_name: default
      chunk_size: 512
      batch_size: 10

### CLI equivalent

Once the profile is defined in `config/workflows.yaml` (using `${paths.data_root}` for portability),
the entire pipeline becomes a single CLI command:

```bash
# Dry-run — see what will happen
uv run cli workflow run anonymize_and_ingest_docs --dry-run

# Execute
uv run cli workflow run anonymize_and_ingest_docs

# Override source dir at runtime
uv run cli workflow run anonymize_and_ingest_docs --set base_dir=/path/to/new/docs
```

### When to use Pydantic vs YAML

| | Pydantic (notebook/script) | YAML profile (CLI/CI) |
|--|--|--|
| IDE completion | ✅ | ❌ |
| Runtime validation | ✅ | Partial |
| Programmatic transforms | ✅ | ❌ |
| Repeatable CLI invocation | ❌ | ✅ |
| Config var interpolation (`${paths.*}`) | ❌ | ✅ |
| Version-control friendly | Both | Both |

Use Pydantic during exploration; commit a YAML profile once the pipeline is stable.

## 7. Architecture summary

```
                    ┌─────────────────────────────────────────┐
                    │          anonymize_text()                │
                    │  (genai_tk.workflow.anonymization.core)  │
                    └───────────────┬─────────────────────────┘
                                    │ called by
              ┌─────────────────────┴──────────────────────┐
              │                                             │
  ┌───────────▼───────────────┐           ┌────────────────▼──────────────┐
  │  anonymize_files_flow     │           │  AnonymizationMiddleware       │
  │  (ETL / batch / Prefect)  │           │  (agent runtime / LangChain)   │
  │                           │           │                                │
  │  File → anonymized file   │           │  HumanMessage → anonymized msg │
  │  + manifest.json          │           │  AIMessage    → deanonymized   │
  └───────────┬───────────────┘           └────────────────────────────────┘
              │
              ▼
  ┌───────────────────────────┐
  │  rag_file_ingestion_flow  │
  │  (chunk + embed + store)  │
  └───────────────────────────┘
```

**Key insight**: PII scrubbing can happen at two points:

1. **ETL time** (this flow) — data is anonymized *before* it enters the vector store. Safer for compliance. The RAG system never sees real PII.
2. **Agent runtime** (`AnonymizationMiddleware`) — anonymization is transparent to the user; the agent sees fake values and its responses are deanonymized before delivery.

The two strategies can be combined: anonymize at ETL, then add the middleware as a second layer for any remaining PII that may appear in user queries.